# Prepare UBI Training Data for Azure ML

**Purpose**: Load data from Fabric gold tables, engineer features and targets, and save to lakehouse for Azure ML consumption.

**Output**: `gold_ml_training_data` table with:
- 8 feature columns
- 3 target columns (risk_factor, expected_loss_cost, risk_score)
- Policy identifiers for joining

**Next Step**: Use the Azure ML notebook to read this data, train, register, and deploy the model.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

## Step 1: Load and Join Gold Tables

In [ ]:
# Load policy period features with baseline
ppf = spark.table("gold_policy_period_features")
loss = spark.table("gold_policy_period_loss")

# Calculate baseline ELC by coverage type
baseline = (
    ppf.join(loss, ["policy_number", "policy_start_date", "policy_end_date"], "left")
    .groupBy("coverage_type")
    .agg(F.avg("total_payout_amount").alias("baseline_elc"))
    .fillna(0, ["baseline_elc"])
)

# Join features with baseline and actual loss
training_data = (
    ppf.join(baseline, "coverage_type", "left")
    .join(loss, ["policy_number", "policy_start_date", "policy_end_date"], "left")
    .fillna(0, ["total_payout_amount", "baseline_elc"])
)

print(f"Loaded {training_data.count()} policy periods for training")
display(training_data.limit(5))

## Step 2: Calculate Target Variables

In [ ]:
# Calculate target variables as Spark columns
training_data_with_targets = training_data.withColumn(
    "target_expected_loss_cost",
    F.col("total_payout_amount").cast(DoubleType())
).withColumn(
    "target_risk_factor",
    F.when(
        F.col("baseline_elc") > 0,
        F.col("total_payout_amount") / F.col("baseline_elc")
    ).otherwise(1.0)
).withColumn(
    "target_risk_score",
    (
        F.lit(50.0) +
        F.lit(2.0) * F.coalesce(F.col("speeding_per_100_miles"), F.lit(0.0)) +
        F.lit(3.0) * F.coalesce(F.col("harsh_events_per_100_miles"), F.lit(0.0)) +
        F.lit(20.0) * F.coalesce(F.col("night_miles_share"), F.lit(0.0)) -
        F.lit(0.5) * F.coalesce(F.col("avg_safety_score"), F.lit(0.0))
    )
).withColumn(
    "target_risk_score",
    F.when(F.col("target_risk_score") < 0, 0)
     .when(F.col("target_risk_score") > 100, 100)
     .otherwise(F.col("target_risk_score"))
)

print("✓ Target variables calculated")
display(training_data_with_targets.select(
    "policy_number",
    "target_risk_factor",
    "target_expected_loss_cost",
    "target_risk_score"
).limit(5))

## Step 3: Select Features and Targets for ML Training

In [ ]:
# Select feature columns and target columns
ml_training_data = training_data_with_targets.select(
    # Identifiers (for future joins if needed)
    "policy_number",
    "policy_start_date",
    "policy_end_date",
    "coverage_type",
    
    # Features (8 columns)
    "speeding_per_100_miles",
    "harsh_events_per_100_miles",
    "night_miles_share",
    "avg_safety_score",
    "total_trips",
    "total_miles",
    "high_risk_trip_share",
    "baseline_elc",
    
    # Targets (3 columns)
    "target_risk_factor",
    "target_expected_loss_cost",
    "target_risk_score"
).fillna(0)

print(f"ML training dataset shape: {ml_training_data.count()} rows")
print(f"Columns: {len(ml_training_data.columns)}")
display(ml_training_data.limit(10))

## Step 4: Save to Lakehouse Table

In [ ]:
# Save as Delta table in lakehouse
table_name = "gold_ml_training_data"

ml_training_data.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"✓ Training data saved to table: {table_name}")
print(f"✓ Row count: {ml_training_data.count()}")
print(f"\n{'='*80}")
print("NEXT STEPS:")
print("="*80)
print("Run Step 5 below to export data to Files section for Azure ML access")
print("="*80)

## Step 5: Export to Files Section for Azure ML

**Important**: Azure ML can only access the Files section of a lakehouse, not Tables.

Export the training data table to Files as CSV format.

In [ ]:
# Export training data to Files section as a single CSV file
output_path = "Files/ml_data/training_data.csv"

print(f"Exporting training data to: {output_path}")

# Convert to Pandas and write as single CSV file
df_pandas = spark.table(table_name).toPandas()
df_pandas.to_csv(output_path, index=False)

print(f"✓ Data exported successfully!")
print(f"\nFile location: {output_path}")
print(f"Rows exported: {len(df_pandas):,}")
print(f"\n{'='*80}")
print("AZURE ML SETUP:")
print("="*80)
print("1. Open Azure ML Studio (ml.azure.com)")
print("2. Navigate to your workspace (sbazureml)")
print("3. Create a compute instance (if not already created)")
print("4. Upload and run the azureml_train_deploy_model.ipynb notebook")
print("5. The notebook will:")
print("   - Create a datastore connection to this lakehouse")
print(f"   - Read from: {output_path}")
print("   - Train, register, and deploy the model")
print("="*80)